In [3]:
from md_Helpers import delete_run

run_id = "20260904190252"

delete_run(run_id)

{'run_id': '20260904190252',
 'sim_type': 'Thermalization',
 'status': 'Cancelled',
 'run_directory': '/exp/e961/data/MDsims-data/pnichols/SQL/Thermalization/20260904190252',
 'trajectory_gsd': '/exp/e961/data/MDsims-data/pnichols/SQL/Thermalization/20260904190252/trajectory.gsd',
 'trajectory_exists': True,
 'run_hdf5': '/exp/e961/data/MDsims-data/pnichols/SQL/Thermalization/20260904190252/run.hdf5',
 'hdf5_exists': True,
 'master_rows': 1,
 'thermalization_rows': 0,
 'dry_run': True}

In [4]:
delete_run(
    run_id,
    dry_run=False,
    confirm_run_id=run_id,
)

{'run_id': '20260904190252',
 'sim_type': 'Thermalization',
 'status': 'Cancelled',
 'run_directory': '/exp/e961/data/MDsims-data/pnichols/SQL/Thermalization/20260904190252',
 'trajectory_gsd': '/exp/e961/data/MDsims-data/pnichols/SQL/Thermalization/20260904190252/trajectory.gsd',
 'trajectory_exists': True,
 'run_hdf5': '/exp/e961/data/MDsims-data/pnichols/SQL/Thermalization/20260904190252/run.hdf5',
 'hdf5_exists': True,
 'master_rows': 1,
 'thermalization_rows': 0,
 'dry_run': False,
 'thermalization_rows_deleted': 0,
 'master_rows_deleted': 1,
 'directory_deleted': True}

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import shutil

from md_Helpers import ProjectPaths, SQLiteRunDatabase

# Exact remote location shown by your terminal output
EXPECTED_TOP = Path("/exp/e961/data/MDsims-data/pnichols/SQL").resolve()

paths = ProjectPaths()
top = paths.top_directory.resolve()
thermal_directory = (top / "Thermalization").resolve()
database_path = paths.database.resolve()

# ============================================================
# Safety checks
# ============================================================

if top != EXPECTED_TOP:
    raise RuntimeError(
        f"Refusing reset.\nExpected: {EXPECTED_TOP}\nFound:    {top}"
    )

if thermal_directory.parent != top:
    raise RuntimeError(f"Unsafe Thermalization path: {thermal_directory}")

if not database_path.exists():
    raise FileNotFoundError(f"Database not found: {database_path}")

database = SQLiteRunDatabase(database_path)

with database.connection() as connection:
    master_before = connection.execute(
        "SELECT COUNT(*) FROM MD_Master"
    ).fetchone()[0]

    thermal_before = connection.execute(
        "SELECT COUNT(*) FROM Thermalization"
    ).fetchone()[0]

print("Deleting:")
print("  Directory:", thermal_directory)
print("  Database:", database_path)
print("  Master rows:", master_before)
print("  Thermalization rows:", thermal_before)

# ============================================================
# Temporarily move files so they can be restored if SQL fails
# ============================================================

quarantine = top / (
    ".Thermalization_reset_"
    + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
)

if quarantine.exists():
    raise FileExistsError(f"Temporary reset directory exists: {quarantine}")

if thermal_directory.exists():
    thermal_directory.rename(quarantine)

thermal_directory.mkdir(parents=True, exist_ok=False)

# ============================================================
# Clear child table first, then Master
# ============================================================

try:
    with database.connection() as connection:
        connection.execute("DELETE FROM Thermalization")
        connection.execute("DELETE FROM MD_Master")
except Exception:
    # Restore files if the database transaction fails
    thermal_directory.rmdir()

    if quarantine.exists():
        quarantine.rename(thermal_directory)

    raise

# SQL commit succeeded, so permanently remove old run files
if quarantine.exists():
    shutil.rmtree(quarantine)

# ============================================================
# Verify
# ============================================================

with database.connection() as connection:
    master_after = connection.execute(
        "SELECT COUNT(*) FROM MD_Master"
    ).fetchone()[0]

    thermal_after = connection.execute(
        "SELECT COUNT(*) FROM Thermalization"
    ).fetchone()[0]

remaining_files = list(thermal_directory.iterdir())

print()
print("Reset complete.")
print("Master rows:", master_after)
print("Thermalization rows:", thermal_after)
print("Remaining run entries:", len(remaining_files))

Deleting:
  Directory: /exp/e961/data/MDsims-data/pnichols/SQL/Thermalization
  Database: /exp/e961/data/MDsims-data/pnichols/SQL/mdsims.sqlite3
  Master rows: 2
  Thermalization rows: 2

Reset complete.
Master rows: 0
Thermalization rows: 0
Remaining run entries: 0
